In [ ]:
with open("/content/hostel_bois.txt", 'r', encoding='utf-8') as f:
    lines = f.readlines()

messages = []
system_count = 0
media_count = 0
deleted_count = 0
group_name = "Unknown"

for line in lines:
    line = line.strip()
    if line == '':
        continue

    if not (line[2] == '/' and line[5] == '/'):
        if messages:
            messages[-1]['text'] += ' ' + line
        continue


    parts = line.split(' - ', 1)
    rest = parts[1]
    if ': ' in rest:
        sender, message = rest.split(': ', 1)
        if message == '<Media omitted>':
            media_count += 1
        elif message == 'This message was deleted':
            deleted_count += 1
        else:
            messages.append({'timestamp': parts[0], 'sender': sender, 'text': message})
    else:
        system_count += 1
        if 'created group' in rest:
            group_name = rest.split('"')[1]


msg_count = {}

for msg in messages:
    sender = msg['sender']
    if sender in msg_count:
        msg_count[sender] += 1
    else:
        msg_count[sender] = 1

total = sum(msg_count.values())
sorted_count = sorted(msg_count.items(), key=lambda x: x[1], reverse=True)

first_date = messages[0]['timestamp'].split(',')[0]
last_date = messages[-1]['timestamp'].split(',')[0]

print("=" * 60)
print("GROUP OVERVIEW")
print("=" * 60)
print(f"{'Group':<15}: {group_name}")
print(f"{'Period':<15}: {first_date} to {last_date}")
print(f"{'Total messages':<15}: {total}")
print(f"{'Participants':<15}: {len(msg_count)}")
print()
print("MESSAGES PER PERSON")
for name, count in sorted_count:
    percent = (count / total) * 100
    print(f"{name:<10}: {count} ({percent:.1f}%)")


stop_words = ['i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for',
              'it', 'my', 'me', 'you', 'he', 'she', 'we', 'they', 'this', 'that',
              'was', 'are', 'be', 'at', 'an', 'so', 'do', 'if', 'up', 'but', 'not',
              'how', 'am', 'his', 'have', 'just', 'about', 'which', 'with', 'your',
              'will', 'has', 'had', 'been', 'from', 'what', 'when', 'who', 'its',
              'no', 'yes', 'ok', 'okay', 'hi', 'hey', 'haha', 'like', 'get', 'got']

word_count = {}

for msg in messages:
    words = msg['text'].lower().split()
    for word in words:
        word = word.strip('.,!?":;)(')
        if word == '' or word in stop_words:
            continue
        if word in word_count:
            word_count[word] += 1
        else:
            word_count[word] = 1

top_words = sorted(word_count.items(), key=lambda x: x[1], reverse=True)[:10]


print("=" * 60)
print("THIS GROUP'S FAVOURITE WORDS")
print("=" * 60)

max_count = top_words[0][1]

for word, count in top_words:
    bar_length = int((count / max_count) * 20)
    bar = '█' * bar_length
    print(f"{word:<15} {bar:<22} {count}")
    print()

import numpy as np

participants = list(msg_count.keys())

matrix = np.zeros((len(participants), 24), dtype=int)
for msg in messages:
    hour = int(msg['timestamp'].split(',')[1].strip().split(':')[0])
    person_index = participants.index(msg['sender'])
    matrix[person_index][hour] += 1

print("=" * 60)
print("ACTIVITY HEATMAP (messages by hour)")
print("=" * 60)
print(f"{'':10}", end='')
for h in range(0, 24, 3):
    print(f"{h:02d}  ", end='')
print()

for i, name in enumerate(participants):
    row = matrix[i]
    row_max = row.max()
    print(f"{name:<10}", end='')
    for h in range(0, 24, 3):
        val = row[h]
        if row_max == 0:
            block = '.  '
        elif val <= row_max * 0.25:
            block = '.  '
        elif val <= row_max * 0.50:
            block = '░  '
        elif val <= row_max * 0.75:
            block = '▒  '
        else:
            block = '█  '
        print(block, end=' ')
    print()

from datetime import datetime

max_gap = 0
max_gap_start = None
max_gap_end = None

for i in range(1, len(messages)):
    dt1 = datetime.strptime(messages[i-1]['timestamp'], '%d/%m/%y, %H:%M')
    dt2 = datetime.strptime(messages[i]['timestamp'], '%d/%m/%y, %H:%M')
    gap = (dt2 - dt1).total_seconds() / 3600

    if gap > max_gap:
        max_gap = gap
        max_gap_start = messages[i-1]['timestamp']
        max_gap_end = messages[i]['timestamp']

response_times = {name: [] for name in participants}

for i in range(1, len(messages)):
    prev = messages[i-1]
    curr = messages[i]
    if curr['sender'] != prev['sender']:
        dt1 = datetime.strptime(prev['timestamp'], '%d/%m/%y, %H:%M')
        dt2 = datetime.strptime(curr['timestamp'], '%d/%m/%y, %H:%M')
        gap = (dt2 - dt1).total_seconds() / 60
        if 0 < gap < 60:
            response_times[curr['sender']].append(gap)

print("=" * 60)
print("RESPONSE TIMES (average minutes to reply)")
print("=" * 60)
for name in participants:
    times = response_times[name]
    if times:
        avg = sum(times) / len(times)
        print(f"{name:<10}: {avg:.1f} mins")

print()
print(f"Longest silent streak : {max_gap:.1f} hours")
print(f"From : {max_gap_start}")
print(f"To   : {max_gap_end}")

night_msgs = {name: 0 for name in participants}

for msg in messages:
    hour = int(msg['timestamp'].split(',')[1].strip().split(':')[0])
    if hour >= 22 or hour <= 5:
        night_msgs[msg['sender']] += 1

msg_lengths = {name: [] for name in participants}

for msg in messages:
    msg_lengths[msg['sender']].append(len(msg['text'].split()))

avg_lengths = {}
for name in participants:
    lengths = msg_lengths[name]
    avg_lengths[name] = sum(lengths) / len(lengths)


comedy_words = ['lol', 'haha', 'hehe', 'lmao', '😂', 'xD', 'hahaha']
comedy_count = {name: 0 for name in participants}

for msg in messages:
    text = msg['text'].lower()
    for word in comedy_words:
        if word in text:
            comedy_count[msg['sender']] += 1

archetypes = {}

def assign(name, title):
    if name not in archetypes:
        archetypes[name] = title

night_ratio = {name: night_msgs[name] / msg_count[name] for name in participants}
assign(max(night_ratio, key=lambda x: night_ratio[x]), '🦉 Night Owl')
assign(max(msg_count, key=lambda x: msg_count[x]), '📢 Spammer')
assign(min(msg_count, key=lambda x: msg_count[x]), '👻 Ghost')
assign(max(avg_lengths, key=lambda x: avg_lengths[x]), '🧠 Philosopher')
assign(max(comedy_count, key=lambda x: comedy_count[x]), '😂 Comedian')

avg_response = {name: sum(response_times[name])/len(response_times[name]) for name in participants if response_times[name]}
assign(min(avg_response, key=lambda x: avg_response[x]), '⚡ Quick Responder')
assign(max(avg_response, key=lambda x: avg_response[x]), '🐢 Slow Responder')

for name in participants:
    if name not in archetypes:
        archetypes[name] = '🕵️ Lurker'

print("=" * 60)
print("PERSONALITY ARCHETYPES")
print("=" * 60)
for name in participants:
    print(f"{name:<10} : {archetypes[name]}")


print("=" * 60)
print("        GROUPDNA REPORT")
print(f"        {group_name}")
print("=" * 60)
print(f"Period : {first_date} to {last_date}")
print(f"Total Messages : {len(messages)}")
print(f"Participants   : {len(participants)}")
print(f"Media Shared   : {media_count}")
print(f"Deleted Msgs   : {deleted_count}")
print(f"System Msgs    : {system_count}")

print("=" * 60)
print("        END OF GROUPDNA REPORT")
print("=" * 60)

GROUP OVERVIEW
Group          : Hostel Bois 4ever
Period         : 01/04/24 to 30/05/24
Total messages : 3127
Participants   : 6

MESSAGES PER PERSON
Rahul     : 940 (30.1%)
Priya     : 712 (22.8%)
Neha      : 624 (20.0%)
Aman      : 484 (15.5%)
Karan     : 345 (11.0%)
Vikas     : 22 (0.7%)
THIS GROUP'S FAVOURITE WORDS
guys            ████████████████████   318

hai             ████████████████       268

today           ████████████████       257

everyone        ███████████            187

telling         ███████████            179

bhai            ██████████             160

one             █████████              157

started         █████████              150

scene           █████████              145

entire          █████████              145

ACTIVITY HEATMAP (messages by hour)
          00  03  06  09  12  15  18  21  
Rahul     .   .   .   .   ▒   ▒   █   █   
Priya     .   .   .   █   █   ░   ▒   ░   
Karan     .   .   .   ░   █   ▒   ▒   ░   
Neha      .   .   .   █   ▒   .